# N25 — Held-out Facial Action Unit (AU) evaluation

**Supplementary evaluation; NO diffusion generation or re-training.** Frozen N23 image outputs (108 sources × 3 conditions) and 108 original images. Outputs remain in **private Google Drive**. This uses [Py-Feat Detectorv2](https://py-feat.org/basic_tutorials/Detecting_Images/) to measure AU predictions; these are estimated facial action units, **not human FACS annotations or proof of clinical fidelity**.

**Protocol:** one original and three outputs per source; strictly pair by source ID; preserve failures; compare AU vectors by mean absolute error (MAE), AU vector cosine and per-AU absolute shifts; calculate paired bootstrap CIs by independently resampling **source IDs**. Results exploratory/post hoc, not a preregistered endpoint. Do not publish face images or source-level CSVs.

**Colab instructions:** Runtime → Change runtime type → GPU (recommended), then Runtime → Run all. Drive mount needs your confirmation. Re-running resumes saved per-image AU predictions and checks source hashes. Download the final aggregate CSV and JSON, not the private per-image table.

In [1]:
%pip -q install "py-feat>=0.6" "pandas>=2.2,<3" "numpy>=1.26,<3" "Pillow>=10" "scipy>=1.11" "tqdm>=4.66"

from google.colab import drive
drive.mount('/content/drive')

import sys, platform, importlib.metadata, hashlib, json, re, traceback, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07_SOURCES = ROOT / '20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664' / 'sources'
# Preferred N23 directory inferred from existing verified run organization:
N23_RUN = ROOT / 'nb23_final_heldout_regeneration_frozen_v1'
N23_IMAGES = N23_RUN / 'images'
# You can also use the exact N23 run image path if it differs:
# N23_IMAGES = Path('/content/drive/MyDrive/thesis_all/outputs/runs/YOUR_N23_RUN/generated_images')
OUT = ROOT / 'nb25_heldout_au_evaluation_v1'
OUT.mkdir(parents=True, exist_ok=True)

print('Python:', platform.python_version(), '| py-feat:', importlib.metadata.version('py-feat'))
print('Source folder:', N07_SOURCES, N07_SOURCES.is_dir())
print('Generated folder:', N23_IMAGES, N23_IMAGES.is_dir())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 25.3 MB/s eta 0:00:00
Mounted at /content/drive
Python: 3.12.13 | py-feat: 2.1.5
Source folder: /content/drive/MyDrive/thesis_all/outputs/runs/20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664/sources True
Generated folder: /content/drive/MyDrive/thesis_all/outputs/runs/nb23_final_heldout_regeneration_frozen_v1/images True


In [2]:
# Discover the genuine N23 generated image folder; do not fabricate images or change ID membership.
if not N23_IMAGES.is_dir():
    possibilities = [p for p in ROOT.glob('*/generated_images') if any(p.glob('nb23_source_*_n07_g4p5_s42.png'))]
    possibilities += [p for p in ROOT.glob('*/images') if any(p.glob('nb23_source_*_n07_g4p5_s42.png'))]
    possibilities = list(dict.fromkeys(possibilities))
    print('Discovered generated-image folders:', possibilities)
    assert len(possibilities) == 1, 'Cannot uniquely locate N23 image folder. Set N23_IMAGES explicitly to the correct run folder.'
    N23_IMAGES = possibilities[0]
assert N07_SOURCES.is_dir(), 'Missing N07 sources folder: check Drive mount and path.'
assert N23_IMAGES.is_dir(), 'Missing N23 generated images folder.'

CONDITIONS = ['n07_g4p5', 'n07_g5p5', 'same_expression_g4p5']
ids = sorted(set(re.findall(r'^nb23_source_(\d{3})_n07_g4p5_s42\.png$', p.name)[0]
                 for p in N23_IMAGES.glob('nb23_source_*_n07_g4p5_s42.png')
                 if re.fullmatch(r'nb23_source_\d{3}_n07_g4p5_s42\.png',p.name)))
assert len(ids) == 108, f'Expected exactly 108 held-out sources, got {len(ids)}'
expected = []
for sid in ids:
    expected.append(dict(source_id=sid, condition='original', path=N07_SOURCES/f'source_{sid}.png'))
    for c in CONDITIONS:
        expected.append(dict(source_id=sid, condition=c, path=N23_IMAGES/f'nb23_source_{sid}_{c}_s42.png'))
manifest = pd.DataFrame(expected)
missing = manifest.loc[~manifest.path.map(lambda p: p.is_file())]
assert missing.empty, 'Missing expected images, e.g. '+repr(missing.head(5).to_dict('records'))
assert len(manifest)==432 and manifest[['source_id','condition']].duplicated().sum()==0
print(f'PREFLIGHT PASS: {len(ids)} sources, {len(manifest)} images. No outputs are modified.')
print(manifest.groupby('condition').size().to_string())

PREFLIGHT PASS: 108 sources, 432 images. No outputs are modified.
condition
n07_g4p5                108
n07_g5p5                108
original                108
same_expression_g4p5    108


In [3]:
# Py-Feat's Detectorv2 predicts 20 AU outputs. It is an estimated-AU model.
import torch
from feat import Detectorv2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
detector = Detectorv2(device=DEVICE, identity_model=None)

def detect_one(path):
    fex = detector.detect(str(path), data_type='image', batch_size=1, progress_bar=False)
    df = pd.DataFrame(fex)
    au_cols = [c for c in df.columns if re.fullmatch(r'AU\d{1,2}', str(c))]
    if len(df)==0 or not au_cols:
        return {}, 'NO_FACE_OR_AU_COLUMNS', len(df)
    # No unrecorded face selection: ambiguous/multiple faces marked invalid.
    if len(df)!=1:
        return {}, 'AMBIGUOUS_FACE_COUNT', len(df)
    values = pd.to_numeric(df.iloc[0][au_cols],errors='coerce')
    if values.isna().any() or not np.isfinite(values.to_numpy(dtype=float)).all():
        return {}, 'INVALID_AU_VALUES', len(df)
    return {str(k): float(v) for k,v in values.items()}, 'VALID', 1

probe=manifest.iloc[0]
probe_aus,status,nfaces=detect_one(probe.path)
print('Smoke test:',probe.source_id,probe.condition,status,'detected AU columns:',len(probe_aus))
assert status=='VALID' and len(probe_aus)>=10, 'AU smoke test failed. Do not proceed until model/runtime issue is resolved.'
AU_COLUMNS = sorted(probe_aus)
print('Frozen observed AU set:',AU_COLUMNS)

Device: cuda


Smoke test: 002 original VALID detected AU columns: 20
Frozen observed AU set: ['AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10', 'AU11', 'AU12', 'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24', 'AU25', 'AU26', 'AU28', 'AU43']


In [4]:
# Resume-safe per-image extraction. Every record is keyed by condition+source and file SHA-256.
# Per-image table stays PRIVATE. On restart, prior rows are reused only if SHA and AU columns match.
RAW=OUT/'private_au_rows.csv'
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

columns=['source_id','condition','path','sha256','status','n_faces','error']+AU_COLUMNS
old=pd.read_csv(RAW, dtype={'source_id':str}).to_dict('records') if RAW.is_file() else []
cache={(str(r['source_id']).zfill(3),r['condition']):r for r in old}
results=[]
for row in tqdm(manifest.itertuples(index=False),total=len(manifest),desc='Extracting held-out AUs'):
    key=(row.source_id,row.condition)
    sha=digest(row.path)
    cached=cache.get(key)
    if cached and cached.get('sha256')==sha and cached.get('status')=='VALID' and all(pd.notna(cached.get(k)) for k in AU_COLUMNS):
        results.append(cached);continue
    values={};status='ERROR';faces=0;error=''
    try:
        values,status,faces=detect_one(row.path)
        assert not values or set(values)==set(AU_COLUMNS), 'AU model output columns unexpectedly changed'
    except Exception as ex:
        error=f'{type(ex).__name__}: {str(ex)[:300]}'
    rec={'source_id':row.source_id,'condition':row.condition,'path':str(row.path),'sha256':sha,
         'status':status,'n_faces':faces,'error':error,**values}
    results.append(rec)
    # Persist after EACH image: interruption never loses progress.
    pd.DataFrame(results,columns=columns).to_csv(RAW,index=False)
raw=pd.DataFrame(results,columns=columns)
raw.to_csv(RAW,index=False)
print('Coverage by condition:')
print(raw.groupby('condition')['status'].value_counts().to_string())
assert len(raw)==432 and not raw[['source_id','condition']].duplicated().any(), 'Run did not retain 432 unique rows'

Coverage by condition:
condition             status              
n07_g4p5              VALID                   108
n07_g5p5              VALID                   108
original              VALID                   107
                      AMBIGUOUS_FACE_COUNT      1
same_expression_g4p5  VALID                   108


In [5]:
# Paired AU comparisons. Validity gate requires an ORIGINAL+OUTPUT pair to have valid AU vectors.
orig=raw.query("condition == 'original'").set_index('source_id')
paired=[]
for cond in CONDITIONS:
    out=raw.loc[raw.condition==cond].set_index('source_id')
    assert len(out)==108 and out.index.is_unique
    for sid in ids:
        o,v=orig.loc[sid],out.loc[sid]
        valid=bool(o.status=='VALID' and v.status=='VALID')
        record={'source_id':sid,'condition':cond,'pair_valid':valid,
                'original_valid':o.status=='VALID','output_valid':v.status=='VALID'}
        if valid:
            a=o[AU_COLUMNS].to_numpy(dtype=float); b=v[AU_COLUMNS].to_numpy(dtype=float)
            record['au_mae']=float(np.mean(np.abs(a-b)))
            na,nb=np.linalg.norm(a),np.linalg.norm(b)
            record['au_cosine']=float(np.dot(a,b)/(na*nb)) if na>0 and nb>0 else np.nan
            for k in AU_COLUMNS: record['abs_delta_'+k]=float(abs(v[k]-o[k]))
        paired.append(record)
paired=pd.DataFrame(paired)
paired.to_csv(OUT/'private_paired_au_rows.csv',index=False)
print(paired.groupby('condition')[['pair_valid','original_valid','output_valid']].agg(['sum','count']).to_string())

# Source-level paired contrasts only; both outputs AND original must have valid AUs.
wide=paired.pivot(index='source_id',columns='condition',values='au_mae')
contrasts={
  'g5p5_minus_g4p5': ('n07_g5p5','n07_g4p5'),
  'same_expression_minus_g4p5': ('same_expression_g4p5','n07_g4p5'),
  'same_expression_minus_g5p5': ('same_expression_g4p5','n07_g5p5'),
}
# CIs: seeded nonparametric percentile bootstrap resampling independent source IDs.
rng=np.random.default_rng(20261015)
def boot_mean_ci(values,nboot=10000):
    a=np.asarray(values,dtype=float);a=a[np.isfinite(a)]
    if len(a)<2:return (float(np.mean(a)) if len(a) else np.nan,np.nan,np.nan)
    draws=np.array([rng.choice(a,size=len(a),replace=True).mean() for _ in range(nboot)])
    return float(a.mean()),float(np.quantile(draws,.025)),float(np.quantile(draws,.975))

summary=[]
for cond in CONDITIONS:
    frame=paired.loc[paired.condition==cond]
    valid=frame.loc[frame.pair_valid]
    for metric in ['au_mae','au_cosine']:
        mean,lo,hi=boot_mean_ci(valid[metric].dropna())
        summary.append(dict(condition=cond,metric=metric,n_total_sources=108,
                            n_valid_pairs=len(valid),coverage=len(valid)/108,
                            mean=mean,ci95_low=lo,ci95_high=hi,
                            ci_status='complete' if len(valid)==108 else 'conditional_on_valid_pairs'))
    for au in AU_COLUMNS:
        metric='abs_delta_'+au
        mean,lo,hi=boot_mean_ci(valid[metric].dropna())
        summary.append(dict(condition=cond,metric=metric,n_total_sources=108,
                            n_valid_pairs=len(valid),coverage=len(valid)/108,
                            mean=mean,ci95_low=lo,ci95_high=hi,
                            ci_status='complete' if len(valid)==108 else 'conditional_on_valid_pairs'))
aggregate=pd.DataFrame(summary)
aggregate.to_csv(OUT/'public_au_aggregate.csv',index=False)

contr=[]
for label,(left,right) in contrasts.items():
    both=wide[[left,right]].dropna()
    delta=(both[left]-both[right]).to_numpy(dtype=float)
    m,lo,hi=boot_mean_ci(delta)
    contr.append(dict(contrast=label,metric='au_mae',n_paired_sources=len(both),
                      mean_delta=m,ci95_low=lo,ci95_high=hi,
                      ci_status='complete' if len(both)==108 else 'conditional_on_jointly_valid_sources'))
pd.DataFrame(contr).to_csv(OUT/'public_au_paired_contrasts.csv',index=False)

run_status={'analysis':'SUPPLEMENTARY_POSTHOC_AU','n_original_sources':108,'n_generated_images':324,
 'total_images_expected':432,'extracted_images':len(raw),'valid_images':int((raw.status=='VALID').sum()),
 'conditions':CONDITIONS,'model':'Py-Feat Detectorv2','py_feat_version':importlib.metadata.version('py-feat'),
 'AU_columns':AU_COLUMNS,'seed_bootstrap':20261015,'bootstrap_n':10000,
 'limitations':['AU predictions are not human FACS annotations','No clinical validation',
                'If missing detections, comparisons conditional on valid pairs',
                'Exploratory post hoc evaluation; no model generation or training']}
(OUT/'public_au_run_status.json').write_text(json.dumps(run_status,indent=2),encoding='utf8')
print('AGGREGATE RESULTS:')
print(aggregate.loc[aggregate.metric.isin(['au_mae','au_cosine'])].to_string(index=False))
print('PAIRED CONTRASTS:')
print(pd.DataFrame(contr).to_string(index=False))
print('Private files saved in:',OUT)

                     pair_valid       original_valid       output_valid      
                            sum count            sum count          sum count
condition                                                                    
n07_g4p5                    107   108            107   108          108   108
n07_g5p5                    107   108            107   108          108   108
same_expression_g4p5        107   108            107   108          108   108
AGGREGATE RESULTS:
           condition    metric  n_total_sources  n_valid_pairs  coverage     mean  ci95_low  ci95_high                  ci_status
            n07_g4p5    au_mae              108            107  0.990741 0.092026  0.079923   0.104840 conditional_on_valid_pairs
            n07_g4p5 au_cosine              108            107  0.990741 0.902437  0.880287   0.922003 conditional_on_valid_pairs
            n07_g5p5    au_mae              108            107  0.990741 0.127949  0.114308   0.142120 conditional_on_valid

## After Run all

Confirm the last cell prints `extracted_images = 432`, and inspect coverage by condition. **If AU detection coverage is low, do not hide missingness or claim expression preservation.** Share only these files for scientific discussion: `public_au_aggregate.csv`, `public_au_paired_contrasts.csv`, and `public_au_run_status.json`. Keep `private_au_rows.csv`, `private_paired_au_rows.csv`, original images, and executed cells with any image-level information private.

For interpretation: lower AU MAE suggests closer model-estimated AU vectors; it is not proof of unchanged human-perceived expression. Compare condition differences with their paired CIs and detection rates. The `same_expression_g4p5` ablation lacks an explicit emotion label but other uncontrolled edit effects can remain.